In [31]:
import pandas as pd
from matplotlib import pyplot as plt

In [32]:
def add_daily_dates(group):
    group['time'] = pd.to_datetime(group['time'], errors='coerce')
    
    year_month = group['time'].iloc[0].strftime('%Y-%m')
    start_date = pd.to_datetime(year_month + "-01")
    end_date = pd.to_datetime((start_date + pd.offsets.MonthEnd(0)))

    date_range = pd.date_range(start=start_date, end=end_date, freq='D')[:len(group)]
    group['time'] = date_range
    return group

In [33]:
def is_leap_year(year):
    """Check if a given year is a leap year."""
    return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)

def correct_time_column(df):
    df['time'] = pd.to_datetime(df['time'])
    # Create a list to hold the corrected time values
    corrected_time = []

    # Group by year and month
    for period, group in df.groupby(df['time'].dt.to_period('M')):
        year = period.year
        month = period.month

        # Determine the number of days in the month
        if month == 2:  # February
            days_in_month = 29 if is_leap_year(year) else 28
        elif month in [4, 6, 9, 11]:  # April, June, September, November
            days_in_month = 30
        else:  # All other months
            days_in_month = 31

        # Generate sequential days for the month
        day_sequence = pd.date_range(
            start=f"{year}-{month:02d}-01", periods=days_in_month, freq='D'
        )

        # Trim or repeat the sequence to match the group size
        if len(group) <= len(day_sequence):
            corrected_time.extend(day_sequence[:len(group)])
        else:
            # Repeat last day if group size exceeds number of days in the month
            corrected_time.extend(day_sequence.tolist() + [day_sequence[-1]] * (len(group) - len(day_sequence)))

    # Assign the corrected time to the dataframe
    if len(corrected_time) != len(df):
        raise ValueError("Corrected time length does not match the DataFrame length.")
    df['time'] = corrected_time
    return df


In [34]:
# Read the CSV file
et_df = pd.read_csv(r'F:\geodata\et_dataset_chenhuiling\et_dataset_yang_station.csv', index_col=0)

# Convert the index to datetime format
et_df.index = pd.to_datetime(et_df.index)

# Filter data for dates after January 1, 2000
et_df = et_df[et_df.index >= '2000-01-01']

# Resample and calculate the mean
et_mon = et_df.resample('ME').mean()
et_mon['year_month'] = et_mon.index.to_period('M').astype(str)
# Output the result
et_df

,dsk,hsg,kq,slglk,tgzlk,wlwt,xhl
time,,,,,,,
2000-01-01,3.708333,10.0,17.379310,8.206897,10.88,5.285714,21.600000
2000-01-02,3.333333,5.0,18.258621,9.137931,10.68,5.730159,32.666667
2000-01-03,2.500000,5.0,16.241379,7.862069,10.10,4.984127,32.333333
2000-01-04,1.666667,4.0,13.896552,8.344828,8.98,4.142857,23.266667
2000-01-05,3.583333,2.0,14.293103,8.068966,8.28,3.936508,16.666667
...,...,...,...,...,...,...,...
2020-12-27,1.250000,4.0,34.689655,2.241379,17.84,12.761905,13.266667
2020-12-28,0.916667,1.0,28.637931,2.965517,16.20,10.587302,7.666667
2020-12-29,1.000000,1.0,24.724138,2.827586,9.66,6.126984,10.333333


In [35]:
GCM_name =   ['MPI-ESM1-2-HR'
'EC-Earth3',
 'FGOALS-g3',
'INM-CM5-0',
'INM-CM4-8',
'MRI-ESM2-0',
'BCC-CSM2-MR']

['EC-Earth3',
 'FGOALS-g3',
 'INM-CM5-0',
 'INM-CM4-8',
 'MRI-ESM2-0',
 'BCC-CSM2-MR']

In [36]:
hist_CMIP6_tas = pd.read_csv(f"J:\\CMIP6_r1i1p1f1\\variable_model_df\\tas_day_{GCM_name}_historical_r1i1p1f1.csv")
hist_CMIP6_pr  = pd.read_csv(f"J:\\CMIP6_r1i1p1f1\\variable_model_df\\pr_day_{GCM_name}_historical_r1i1p1f1.csv")

In [37]:
# Apply the correction
hist_CMIP6_tas = correct_time_column(hist_CMIP6_tas)
hist_CMIP6_pr = correct_time_column(hist_CMIP6_pr)
hist_CMIP6_tas= hist_CMIP6_tas[hist_CMIP6_tas['time']>=pd.to_datetime('2000-01-01')]
hist_CMIP6_pr = hist_CMIP6_pr[hist_CMIP6_pr['time']>=pd.to_datetime('2000-01-01')]


In [38]:
CDMet_pr_df = pd.read_csv(r"H:\CDMet\CDMet_pr_combined_df.csv", index_col=0)
CDMet_tm_df = pd.read_csv(r"H:\CDMet\CDMet_tm_combined_df.csv", index_col=0)
CDMet_pr_df['time'] = pd.date_range(start='2000-01-01', end='2020-12-31', freq='D')
CDMet_tm_df['time'] = pd.date_range(start='2000-01-01', end='2020-12-31', freq='D')
CDMet_pr_df.set_index('time', inplace=True)
CDMet_tm_df.set_index('time', inplace=True)
CDMet_pr_df.columns = [x+"pr_CDMet" for x in CDMet_pr_df.columns]
CDMet_tm_df.columns = [x+"tm_CDMet" for x in CDMet_tm_df.columns]
CDMet_pr_df

,dskpr_CDMet,hsgpr_CDMet,kqpr_CDMet,slglkpr_CDMet,tgzlkpr_CDMet,wlwtpr_CDMet,xhlpr_CDMet
time,,,,,,,
2000-01-01,0.130556,0.096703,0.0,0.025,0.0,0.0,0.075
2000-01-02,1.340278,0.158242,0.0,0.000,0.0,0.0,0.000
2000-01-03,1.628056,0.323297,0.0,0.000,0.0,0.0,1.860
2000-01-04,0.000000,0.000000,0.0,0.000,0.0,0.0,0.000
2000-01-05,0.000000,0.000000,0.0,0.000,0.0,0.0,0.000
...,...,...,...,...,...,...,...
2020-12-27,0.000000,0.000000,0.0,0.000,0.0,0.0,0.000
2020-12-28,0.000000,0.000000,0.0,0.000,0.0,0.0,0.000
2020-12-29,0.000000,0.000000,0.0,0.000,0.0,0.0,0.000


In [39]:
date_range = pd.date_range(start='2000-01-01', end='2100-12-31', freq='D')
# Remove the 29th of February
date_range = date_range[~((date_range.month == 2) & (date_range.day == 29))]
# Convert to DataFrame if needed
time_series_df = pd.DataFrame(date_range, columns=['date'])

In [40]:
## Here we do not consider hsg for lack of CMIP6 dataset
file_name_list = ['1_hsg_imputMF','2_dsk_imputMF','3_xhl_imputMF','4_slglk_imputMF','5_kq_imputMF','6_wlwt_imputMF','7_tgzlk_imputMF']
# file_name_list = [ '2_dsk_imputMF','3_xhl_imputMF','4_slglk_imputMF','5_kq_imputMF','6_wlwt_imputMF','7_tgzlk_imputMF']

In [41]:
for GCM_name in [ 'FGOALS-g3', 'MPI-ESM1-2-HR', 'EC-Earth3','BCC-CSM2-MR', 'MRI-ESM2-0', 'INM-CM5-0', 'INM-CM4-8']:
    for file_name in file_name_list:
        abbre = file_name.split('_')[1]
        for scenario in ['ssp126','ssp245','ssp370','ssp585']:
            
            # load the future climate dataset
            future_CMIP6_tas = pd.read_csv(f"J:\\CMIP6_r1i1p1f1\\variable_model_df\\tas_day_{GCM_name}_{scenario}_r1i1p1f1.csv")
            future_CMIP6_pr  = pd.read_csv(f"J:\\CMIP6_r1i1p1f1\\variable_model_df\\pr_day_{GCM_name}_{scenario}_r1i1p1f1.csv")
    
            tas_df = pd.concat([hist_CMIP6_tas,future_CMIP6_tas])
            pr_df = pd.concat([hist_CMIP6_pr,future_CMIP6_pr])
            
            tas_df = correct_time_column(tas_df)
            pr_df = correct_time_column(pr_df)
            
            tas_df['time'] = pd.to_datetime(tas_df['time'])
            pr_df['time'] = pd.to_datetime(pr_df['time'])
            
            tas_df.reset_index(drop=True, inplace=True)
            pr_df.reset_index(drop=True, inplace=True)
            
            # tas_df['time'] =time_series_df
            # pr_df['time'] =time_series_df
            # tas_df = tas_df.groupby("time", group_keys=False).apply(add_daily_dates)
            # pr_df = pr_df.groupby("time", group_keys=False).apply(add_daily_dates)
        
            tas_df.set_index('time', inplace=True)
            pr_df.set_index('time', inplace=True)
            
            # tas_df -= 273.15 # Convert to Celsius temperature
            
    
            
            # load historical runoff observed dataset
            print(file_name,scenario)
            station_daily_df = pd.read_csv(f'F:\\geodata\\river_runoff_obs\\daily_runoff_obs.csv')
            station_daily_df = station_daily_df[['time',abbre]]
            station_daily_df.rename(columns={abbre:'dis'}, inplace=True)
           
            # station_daily_df=station_daily_df[['time','dis','tm','pre']]
            station_daily_df['time'] = pd.to_datetime(station_daily_df['time'])
            station_daily_df.set_index('time', inplace=True)
            
            # station_daily_df = pd.merge(station_daily_df,tas_df[abbre],left_index=True,right_index=True,how='right',suffixes=('','tm'))
            # station_daily_df = pd.merge(station_daily_df, pr_df[abbre],left_index=True,right_index=True,how='right',suffixes=('','pre'))
            # 
            station_daily_tas_pr_df = pd.merge(tas_df[abbre],pr_df[abbre],left_index=True,right_index=True,how='outer',suffixes=('tm_CMIP6','pre_CMIP6'))
            station_daily_df=station_daily_df.merge(station_daily_tas_pr_df,how = 'right',left_index=True,right_index=True)
            '''merge the CDMet for bias correction'''
            station_daily_df = station_daily_df.merge(CDMet_pr_df[abbre+"pr_CDMet"],how='left',left_index=True,right_index=True)
            station_daily_df = station_daily_df.merge(CDMet_tm_df[abbre+"tm_CDMet"],how='left',left_index=True,right_index=True)
            station_daily_df.columns = ['dis',  'tm_CMIP6',  'pre_CMIP6',  'pr_CDMet',  'tm_CDMet']

            # Ensure the time index is in datetime format
            station_daily_df.index = pd.to_datetime(station_daily_df.index)
            station_daily_df = station_daily_df.loc['2000-01-01':'2100-12-31']
            
            # Step 1: Compute monthly mean values for the reference period (1980-2010)
            ref_period = station_daily_df.loc['2000-01-01':'2100-12-31']
            monthly_means_CDMet_tm = station_daily_df.groupby(ref_period.index.month)['tm_CDMet'].mean()
            monthly_means_CMIP6_tm = station_daily_df.groupby(ref_period.index.month)['tm_CMIP6'].mean()
            monthly_means_CDMet_pr = station_daily_df.groupby(ref_period.index.month)['pr_CDMet'].mean()
            monthly_means_CMIP6_pr = station_daily_df.groupby(ref_period.index.month)['pre_CMIP6'].mean()
            
            monthly_Phi_tm = station_daily_df.groupby(ref_period.index.month)['tm_CDMet'].std() / station_daily_df.groupby(ref_period.index.month)['tm_CMIP6'].std()
            monthly_Phi_pr = station_daily_df.groupby(ref_period.index.month)['pr_CDMet'].std() / station_daily_df.groupby(ref_period.index.month)['pre_CMIP6'].std()
            monthly_alpha_tm = monthly_means_CDMet_tm / monthly_means_CMIP6_tm
            monthly_alpha_pr = monthly_means_CDMet_pr / monthly_means_CMIP6_pr         
            
            # Step 3: Apply the biases to the GCM series for the entire period (1980–2100)
            # Create new columns for corrected values
            station_daily_df['tm'] = station_daily_df['tm_CMIP6']  # Initialize with original values
            station_daily_df['pre'] = station_daily_df['pre_CMIP6']  # Initialize with original values
            
            # Apply biases
            for month in range(1, 13):
                month_mask = station_daily_df.index.month == month
                
                """Correct by Hock 2015""" # https://www.frontiersin.org/journals/earth-science/articles/10.3389/feart.2015.00054/full#B26                
                station_daily_df.loc[month_mask, 'tm'] = monthly_means_CDMet_tm[month] + (station_daily_df.loc[month_mask,'tm_CMIP6']-monthly_means_CDMet_tm[month]) * monthly_Phi_tm[month]
                
                # '''Correct by ''' # https://link.springer.com/article/10.1007/s10584-011-0143-4#Equ7
                # station_daily_df.loc[month_mask, 'tm'] = (station_daily_df.loc[month_mask,'tm_CMIP6']-monthly_means_CDMet_tm[month]) * monthly_alpha_tm[month] + (station_daily_df.loc[month_mask,'tm_CMIP6'] * monthly_means_CDMet_tm[month])
                # station_daily_df.loc[month_mask, 'pre'] = (station_daily_df.loc[month_mask,'pre_CMIP6']-monthly_means_CDMet_pr[month]) * monthly_alpha_pr[month] + (station_daily_df.loc[month_mask,'pre_CMIP6'] * monthly_means_CDMet_pr[month])

             
            station_daily_df = station_daily_df.merge(et_df[abbre],left_index=True,right_index=True,how='outer',suffixes=('','ep'))
            station_daily_df.rename(columns={abbre:'ep'}, inplace=True)
            
            # Convert daily to monthly
            station_mon_df = station_daily_df[['dis','tm','pre','ep']].resample('ME').mean()
            station_mon_df['year_month'] = station_mon_df.index.to_period('M').astype(str)
            
    
            '''merge glacier runoff dataset'''
            gr_df = pd.read_csv(f"I:\\GlacierData\\glacier_mon_pre\\R131415_glac_runoff_fixed_monthly_1set_2000_2100-{scenario}-Batch.csv")
            gr_df['time'] = pd.to_datetime(gr_df['time'])
            gr_df['year_month'] = gr_df['time'].dt.to_period('M').astype(str)
            gr_df.set_index('time', inplace=True)
            gr_df.index = gr_df.index.to_period('M').to_timestamp('M')
            
            station_mon_df = pd.merge(station_mon_df,gr_df[[abbre,'year_month']], how='outer', on='year_month',suffixes=('','_gr'))  
            station_mon_df.rename(columns={abbre:'gr','year_month':'time'}, inplace=True)          
            
            station_mon_df[(station_mon_df['time'] >= '2000-01-01') & (station_mon_df['time'] <= '2100-12-31')].to_csv(f'F:\\geodata\\river_runoff_obs\\{file_name}_{GCM_name}_{scenario}_r1i1p1f1_mon.csv')
            station_daily_df[(station_daily_df.index >= '2000-01-01') & (station_daily_df.index <= '2100-12-31')].to_csv(f'F:\\geodata\\river_runoff_obs\\{file_name}_{GCM_name}_{scenario}_r1i1p1f1_daily.csv')

1_hsg_imputMF ssp126
1_hsg_imputMF ssp245
1_hsg_imputMF ssp370
1_hsg_imputMF ssp585
2_dsk_imputMF ssp126
2_dsk_imputMF ssp245
2_dsk_imputMF ssp370
2_dsk_imputMF ssp585
3_xhl_imputMF ssp126
3_xhl_imputMF ssp245
3_xhl_imputMF ssp370
3_xhl_imputMF ssp585
4_slglk_imputMF ssp126
4_slglk_imputMF ssp245
4_slglk_imputMF ssp370
4_slglk_imputMF ssp585
5_kq_imputMF ssp126
5_kq_imputMF ssp245
5_kq_imputMF ssp370
5_kq_imputMF ssp585
6_wlwt_imputMF ssp126
6_wlwt_imputMF ssp245
6_wlwt_imputMF ssp370
6_wlwt_imputMF ssp585
7_tgzlk_imputMF ssp126
7_tgzlk_imputMF ssp245
7_tgzlk_imputMF ssp370
7_tgzlk_imputMF ssp585
1_hsg_imputMF ssp126
1_hsg_imputMF ssp245
1_hsg_imputMF ssp370
1_hsg_imputMF ssp585
2_dsk_imputMF ssp126
2_dsk_imputMF ssp245
2_dsk_imputMF ssp370
2_dsk_imputMF ssp585
3_xhl_imputMF ssp126
3_xhl_imputMF ssp245
3_xhl_imputMF ssp370
3_xhl_imputMF ssp585
4_slglk_imputMF ssp126
4_slglk_imputMF ssp245
4_slglk_imputMF ssp370
4_slglk_imputMF ssp585
5_kq_imputMF ssp126
5_kq_imputMF ssp245
5_kq_imputMF